In [5]:
import os, json, time, gc, warnings
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score,
                             roc_curve, classification_report)

In [6]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✓ GPUs: {len(gpus)}")
else:
    print("⚠ No GPU - using CPU")
tf.keras.mixed_precision.set_global_policy('float32')

# 2. GENERATE / LOAD RECT DATASET (REVISI: min_ratio = 1.3)
def generate_rectangles(n_samples=12000, img_size=28, seed=42, min_ratio=1.3):
    
    rng = np.random.default_rng(seed)
    images = np.zeros((n_samples, img_size, img_size), dtype=np.float32)
    labels = np.zeros(n_samples, dtype=np.int32)

    for i in range(n_samples):
        img = np.ones((img_size, img_size), dtype=np.float32)
        is_wide = rng.integers(0, 2)  # 50% Wide, 50% Tall
        
        if is_wide == 1:
            w = rng.integers(10, 22)
            max_h = max(3, int(w / min_ratio))
            h = rng.integers(3, max_h + 1) if max_h >= 3 else 3
        else:
            h = rng.integers(10, 22)
            max_w = max(3, int(h / min_ratio))
            w = rng.integers(3, max_w + 1) if max_w >= 3 else 3

        y = rng.integers(0, img_size - h + 1)
        x = rng.integers(0, img_size - w + 1)
        
        img[y:y+h, x:x+w] = 0.0
        
        images[i] = img
        labels[i] = is_wide # 1 = Wide, 0 = Tall

    return images, labels

print("GENERATING RECT")

x_full, y_full = generate_rectangles(n_samples=12000, img_size=28, seed=42)
x_full = x_full[..., np.newaxis]

n_total = len(x_full)
n_train = int(n_total * 0.70)
n_val   = int(n_total * 0.15)

rng = np.random.default_rng(42)
perm = rng.permutation(n_total)

x_train_full = x_full[perm[:n_train]]
y_train_full = y_full[perm[:n_train]]
x_val_full   = x_full[perm[n_train:n_train+n_val]]
y_val_full   = y_full[perm[n_train:n_train+n_val]]
x_test_full  = x_full[perm[n_train+n_val:]]
y_test_full  = y_full[perm[n_train+n_val:]]

CLASS_NAMES = ['Tall', 'Wide']
print(f"Train: {x_train_full.shape} | Val: {x_val_full.shape} | Test: {x_test_full.shape}")
print(f"Class distribution — Train: {np.bincount(y_train_full)} | Val: {np.bincount(y_val_full)} | Test: {np.bincount(y_test_full)}")

# 3. AUGMENTATION PIPELINE 
@tf.function
def augment(image, label):
   
    image = tf.image.random_brightness(image, max_delta=0.03)
    return tf.clip_by_value(image, 0.0, 1.0), label

def make_dataset(x, y, batch_size, augment_data=False, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(x), seed=42)
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# 4. CNN BUILDER (REVISI: Dual Global Average Pooling)
def build_cnn_model(hp: dict, num_classes=2):
    num_layers    = int(np.clip(hp['num_layers'],   2, 4))
    filters_base  = int(np.clip(hp['filters_base'], 8, 64))
    dropout_rate  = float(np.clip(hp['dropout_rate'], 0.0, 0.5))
    dense_units   = int(np.clip(hp['dense_units'],  16, 256))
    use_residual  = int(hp.get('use_residual',  0)) == 1
    use_separable = int(hp.get('use_separable', 0)) == 1

    inputs = tf.keras.Input(shape=(28, 28, 1))
    x = inputs

    for i in range(num_layers):
        filters  = min(filters_base * (2 ** i), 128)
        shortcut = x

        if use_separable:
            x = tf.keras.layers.SeparableConv2D(
                    filters, 3, padding='same', use_bias=False,
                    depthwise_regularizer=tf.keras.regularizers.l2(1e-4),
                    pointwise_regularizer=tf.keras.regularizers.l2(1e-4))(x)
        else:
            x = tf.keras.layers.Conv2D(
                    filters, 3, padding='same', use_bias=False,
                    kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation('relu')(x)

        if use_separable:
            x = tf.keras.layers.SeparableConv2D(
                    filters, 3, padding='same', use_bias=False,
                    depthwise_regularizer=tf.keras.regularizers.l2(1e-4),
                    pointwise_regularizer=tf.keras.regularizers.l2(1e-4))(x)
        else:
            x = tf.keras.layers.Conv2D(
                    filters, 3, padding='same', use_bias=False,
                    kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation('relu')(x)

        if use_residual:
            if shortcut.shape[-1] != filters:
                shortcut = tf.keras.layers.Conv2D(
                    filters, 1, padding='same', use_bias=False)(shortcut)
                shortcut = tf.keras.layers.BatchNormalization()(shortcut)
            x = tf.keras.layers.add([x, shortcut])

        x = tf.keras.layers.MaxPooling2D(2, padding='same')(x)
        x = tf.keras.layers.Dropout(dropout_rate)(x)

    # REVISI: Dual GAP (Horizontal & Vertical terpisah) agar model paham rasio spasial
    gap_h = tf.keras.layers.Lambda(lambda t: tf.reduce_mean(t, axis=1, keepdims=True))(x)
    gap_v = tf.keras.layers.Lambda(lambda t: tf.reduce_mean(t, axis=2, keepdims=True))(x)
    
    gap_h = tf.keras.layers.Flatten()(gap_h)
    gap_v = tf.keras.layers.Flatten()(gap_v)
    
    x = tf.keras.layers.Concatenate()([gap_h, gap_v])

    x = tf.keras.layers.Dense(dense_units, use_bias=False,
                               kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

    return tf.keras.Model(inputs, outputs)

# 5. COSINE DECAY CALLBACK
class CosineDecayCallback(tf.keras.callbacks.Callback):
    def __init__(self, lr_start, total_steps):
        super().__init__()
        self.lr_start = lr_start
        self.total_steps = total_steps
        self.step = 0

    def on_train_batch_end(self, batch, logs=None):
        self.step += 1
        frac   = self.step / self.total_steps
        new_lr = max(self.lr_start * 0.5 * (1 + np.cos(np.pi * frac)), 1e-6)
        self.model.optimizer.learning_rate.assign(new_lr)

# 6. EVALUATION FUNCTION (SINGLE GPU)
def evaluate_multi(lion, x_train, y_train, x_val, y_val, epochs=15, verbose=0):
    hp = lion.hyperparameters
    batch_size = int(np.clip(hp['batch_size'], 16, 128))
    initial_lr = float(hp['learning_rate'])

    model = build_cnn_model(hp)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=initial_lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    steps_per_epoch = max(1, len(x_train) // batch_size)
    total_steps     = steps_per_epoch * epochs
    train_ds = make_dataset(x_train, y_train, batch_size, augment_data=True)
    val_ds   = make_dataset(x_val,   y_val,   batch_size, augment_data=False, shuffle=False)

    try:
        t0 = time.time()
        history = model.fit(
            train_ds, epochs=epochs, validation_data=val_ds,
            callbacks=[
                tf.keras.callbacks.EarlyStopping(
                    monitor='val_accuracy', patience=5,
                    restore_best_weights=True, verbose=0),
                CosineDecayCallback(lr_start=initial_lr, total_steps=total_steps)
            ],
            verbose=verbose
        )
        train_time = time.time() - t0
        val_acc = max(history.history['val_accuracy'])
    except Exception as e:
        print(f"    ⚠ Training error: {e}")
        del model; gc.collect(); tf.keras.backend.clear_session()
        return {'accuracy':0.0,'inference_time':999.0,'model_size':999.0,'train_time':0.0}

    n_inf   = min(500, len(x_val))
    inf_idx = np.random.choice(len(x_val), n_inf, replace=False)
    t1 = time.time()
    model.predict(x_val[inf_idx], batch_size=batch_size, verbose=0)
    inf_time = (time.time() - t1) / n_inf * 1000

    model_size = model.count_params() * 4 / (1024**2)
    del model; gc.collect(); tf.keras.backend.clear_session()

    return {
        'accuracy': float(val_acc),
        'inference_time': float(inf_time),
        'model_size': float(model_size),
        'train_time': float(train_time)
    }

# 7. NSGA-II: NORMALIZATION & DOMINANCE
GLOBAL_NORM_BOUNDS = {
    'accuracy':       (0.0,   1.0),
    'inference_time': (0.0, 200.0),
    'model_size':     (0.0, 500.0),
}

def normalize_objectives(pop_fit):
    if not pop_fit:
        return pop_fit
    for p in pop_fit:
        for k in ['accuracy', 'inference_time', 'model_size']:
            mn, mx = GLOBAL_NORM_BOUNDS[k]
            raw = float(np.clip(p[k], mn, mx))
            p[k+'_norm'] = (raw - mn) / (mx - mn) if mx != mn else 0.5
    return pop_fit

def dominates(a, b):
    av = (-a['accuracy_norm'], a['inference_time_norm'], a['model_size_norm'])
    bv = (-b['accuracy_norm'], b['inference_time_norm'], b['model_size_norm'])
    return all(x <= y for x, y in zip(av, bv)) and any(x < y for x, y in zip(av, bv))

def fast_non_dominated_sort(pop):
    if not pop:
        return []
    n = len(pop)
    dc = [0] * n
    dl = [[] for _ in range(n)]
    fronts = [[]]
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if dominates(pop[i], pop[j]):
                dl[i].append(j)
            elif dominates(pop[j], pop[i]):
                dc[i] += 1
        if dc[i] == 0:
            fronts[0].append(i)
    i = 0
    while i < len(fronts) and fronts[i]:
        nf = []
        for idx in fronts[i]:
            for j in dl[idx]:
                dc[j] -= 1
                if dc[j] == 0:
                    nf.append(j)
        if nf:
            fronts.append(nf)
        i += 1
    return [[pop[idx] for idx in f] for f in fronts if f]

def crowding_distance(front):
    n = len(front)
    if n <= 2:
        return [float('inf')] * n
    dists = [0.0] * n
    for obj in ['accuracy_norm', 'inference_time_norm', 'model_size_norm']:
        sidx = sorted(range(n), key=lambda x: front[x][obj])
        dists[sidx[0]] = dists[sidx[-1]] = float('inf')
        span = front[sidx[-1]][obj] - front[sidx[0]][obj]
        if span == 0:
            continue
        for k in range(1, n - 1):
            dists[sidx[k]] += (front[sidx[k+1]][obj] - front[sidx[k-1]][obj]) / span
    return dists

# 8. CLASS LION
class Lion:
    def __init__(self, hyperparameters):
        self.hyperparameters = hyperparameters.copy()

# 9. NS-LSO OPTIMIZER
class NSLSO_Optimizer:
    def __init__(self, population_size=15, max_generations=8, hp_range=None):
        self.pop_size = population_size
        self.max_gen  = max_generations
        self.hp_range = hp_range or {
            'learning_rate': (1e-4, 5e-3),
            'batch_size':    (16,  128),
            'num_layers':    (2,   4),
            'filters_base':  (8,   64),
            'dropout_rate':  (0.0, 0.5),
            'dense_units':   (16,  256),
            'use_residual':  (0,   1),
            'use_separable': (0,   1)
        }
        self.swarm = []
        self.archive = []
        self.history = []

    def initialize(self):
        rng = np.random.default_rng(42)
        self.swarm = []
        for _ in range(self.pop_size):
            hp = {}
            for param, (low, high) in self.hp_range.items():
                if param == 'learning_rate':
                    hp[param] = float(10 ** rng.uniform(np.log10(low), np.log10(high)))
                elif param in ['num_layers', 'batch_size', 'filters_base', 'dense_units',
                               'use_residual', 'use_separable']:
                    hp[param] = int(rng.integers(low, high + 1))
                else:
                    hp[param] = float(rng.uniform(low, high))
            hp['momentum_lr'] = 0.0
            hp['last_acc'] = 0.0
            self.swarm.append(Lion(hp))

    def lion_update(self, lion, archive, beta=0.9, eta=0.1):
        hp = lion.hyperparameters.copy()
        mean_acc = np.mean([p['accuracy'] for p in archive]) if archive else 0.5
        grad = (hp.get('last_acc', 0.5) - mean_acc) / (mean_acc + 1e-8)
        mom = beta * hp.get('momentum_lr', 0.0) + (1 - beta) * grad
        hp['momentum_lr'] = float(mom)
        hp['learning_rate'] = float(np.clip(
            hp['learning_rate'] * (1 + eta * np.sign(mom + 1e-8)), 1e-5, 1e-2))

        if np.random.rand() < 0.3:
            hp['num_layers'] = int(np.clip(
                hp['num_layers'] + np.random.choice([-1, 1]), 2, 4))

        for param in ['filters_base', 'dense_units']:
            if np.random.rand() < 0.3:
                low, high = self.hp_range[param]
                hp[param] = int(np.clip(
                    int(hp[param] * np.random.uniform(0.85, 1.15)), low, high))

        if np.random.rand() < 0.3:
            choices = [16, 32, 64, 128, 256]
            idx = min(range(len(choices)), key=lambda x: abs(choices[x] - int(hp['batch_size'])))
            hp['batch_size'] = choices[int(np.clip(
                idx + np.random.choice([-1, 1]), 0, len(choices) - 1))]

        if np.random.rand() < 0.3:
            hp['dropout_rate'] = float(np.clip(
                hp['dropout_rate'] + np.random.normal(0, 0.05), 0.0, 0.5))

        if np.random.rand() < 0.2:
            hp['use_residual'] = 1 - int(hp.get('use_residual', 0))
        if np.random.rand() < 0.2:
            hp['use_separable'] = 1 - int(hp.get('use_separable', 0))

        return Lion(hp)

    def optimize(self, x_train, y_train, x_val, y_val, epochs_per_eval=15, verbose=1):
        self.initialize()
        best_elite = None
        best_elite_acc = 0.0

        for gen in range(1, self.max_gen + 1):
            if verbose:
          
                print(f"GENERATION {gen}/{self.max_gen}")
               

            pop_fit = []
            for i, lion in enumerate(self.swarm):
                fit = evaluate_multi(lion, x_train, y_train, x_val, y_val,
                                     epochs=epochs_per_eval, verbose=0)
                lion.hyperparameters['last_acc'] = fit['accuracy']

                # REVISI: Menaikkan threshold FAILED menjadi 0.60
                if fit['accuracy'] < 0.60:
                    if verbose:
                        print(f"  Lion {i+1:2d}: FAILED (acc={fit['accuracy']:.4f})")
                    continue

                fit['hyperparams'] = {k: v for k, v in lion.hyperparameters.items()
                                      if not k.startswith('momentum') and k != 'last_acc'}
                pop_fit.append(fit)

                if fit['accuracy'] > best_elite_acc:
                    best_elite_acc = fit['accuracy']
                    best_elite = fit.copy()

                if verbose:
                    print(f"  Lion {i+1:2d}/{self.pop_size}: "
                          f"acc={fit['accuracy']:.4f} | "
                          f"inf={fit['inference_time']:.2f}ms | "
                          f"size={fit['model_size']:.2f}MB")

            if not pop_fit:
                print("  ⚠ All failed! Using backup...")
                fit = evaluate_multi(self.swarm[0], x_train, y_train, x_val, y_val, epochs=20)
                fit['hyperparams'] = {k: v for k, v in self.swarm[0].hyperparameters.items()
                                      if not k.startswith('momentum') and k != 'last_acc'}
                fit['accuracy'] = max(fit['accuracy'], 0.60)
                pop_fit.append(fit)

            pop_fit = normalize_objectives(pop_fit)
            self.archive = normalize_objectives(self.archive)
            fronts = fast_non_dominated_sort(pop_fit + self.archive)

            new_archive = []
            for front in fronts:
                if len(new_archive) + len(front) <= self.pop_size:
                    new_archive.extend(front)
                else:
                    cd = crowding_distance(front)
                    ranked = sorted(zip(cd, front), key=lambda x: x[0], reverse=True)
                    new_archive.extend(s for _, s in ranked[:self.pop_size - len(new_archive)])
                    break
            self.archive = new_archive

            front_0 = fronts[0] if fronts else []
            print(f"\n  >>> FRONT-0: {len(front_0)} solutions <<<")
            for sol in front_0[:3]:
                print(f"      acc={sol['accuracy']:.4f} | "
                      f"inf={sol['inference_time']:.2f}ms | "
                      f"size={sol['model_size']:.2f}MB")

            self.swarm = [self.lion_update(lion, self.archive) for lion in self.swarm]
            if best_elite:
                elite_lion = Lion(best_elite['hyperparams'])
                self.swarm[-1] = elite_lion

            self.history.append({
                'generation': gen,
                'front_0_size': len(front_0),
                'archive_size': len(self.archive),
                'best_acc': max((s['accuracy'] for s in self.archive), default=0.0)
            })

            if gen % 2 == 0:
                self._save_checkpoint(f'/kaggle/working/nslso_rect_checkpoint_gen{gen}.json')

        return self.archive

    def _save_checkpoint(self, path):
        def _cast(v):
            if isinstance(v, (np.floating, float)):
                return float(v)
            if isinstance(v, (np.integer, int)):
                return int(v)
            if isinstance(v, np.ndarray):
                return v.tolist()
            return v
        with open(path, 'w') as f:
            json.dump({
                'archive': [{k: _cast(v) for k, v in s.items() if k != 'hyperparams'}
                            for s in self.archive],
                'history': self.history
            }, f, indent=2)
        print(f"  [✓] Checkpoint saved: {path}")

# 10. FINAL EVALUATION (SINGLE GPU)
def final_test_evaluation(best_hp, x_train_full, y_train_full,
                          x_test, y_test, epochs=30):
    rng = np.random.default_rng(99)
    n = len(x_train_full)
    perm = rng.permutation(n)
    cut = int(n * 0.9)
    x_tr = x_train_full[perm[:cut]]
    y_tr = y_train_full[perm[:cut]]
    x_vl = x_train_full[perm[cut:]]
    y_vl = y_train_full[perm[cut:]]

    batch_size = int(np.clip(best_hp['batch_size'], 16, 128))
    initial_lr = float(best_hp['learning_rate'])

    model = build_cnn_model(best_hp)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=initial_lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    steps_total = max(1, len(x_tr) // batch_size) * epochs
    train_ds = make_dataset(x_tr, y_tr, batch_size, augment_data=True)
    val_ds   = make_dataset(x_vl, y_vl, batch_size, augment_data=False, shuffle=False)

    print(f"\nTraining final model ({epochs} epochs)...")
    history = model.fit(
        train_ds, epochs=epochs, validation_data=val_ds,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor='val_accuracy', patience=10,
                restore_best_weights=True, verbose=1),
            CosineDecayCallback(lr_start=initial_lr, total_steps=steps_total)
        ], verbose=1
    )

    y_pred_probs = model.predict(x_test, batch_size=batch_size, verbose=0).flatten()
    y_pred = (y_pred_probs >= 0.5).astype(np.int32)

    accuracy = accuracy_score(y_test, y_pred)
    precision_pc = precision_score(y_test, y_pred, average=None, zero_division=0)
    recall_pc = recall_score(y_test, y_pred, average=None, zero_division=0)
    f1_pc = f1_score(y_test, y_pred, average=None, zero_division=0)
    precision_macro = float(np.mean(precision_pc))
    recall_macro = float(np.mean(recall_pc))
    f1_macro = float(np.mean(f1_pc))
    precision_weighted = float(precision_score(y_test, y_pred, average='weighted', zero_division=0))
    recall_weighted = float(recall_score(y_test, y_pred, average='weighted', zero_division=0))
    f1_weighted = float(f1_score(y_test, y_pred, average='weighted', zero_division=0))
    roc_auc = float(roc_auc_score(y_test, y_pred_probs))

    cm = confusion_matrix(y_test, y_pred)
    cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100

    model_size = model.count_params() * 4 / (1024**2)
    total_params = model.count_params()

    n_inf = min(500, len(x_test))
    inf_idx = np.random.choice(len(x_test), n_inf, replace=False)
    t1 = time.time()
    model.predict(x_test[inf_idx], batch_size=batch_size, verbose=0)
    inf_time = (time.time() - t1) / n_inf * 1000

    model.save('/kaggle/working/best_rect_model.keras')
    print("[✓] Model saved: /kaggle/working/best_rect_model.keras")

    del model; gc.collect(); tf.keras.backend.clear_session()

    return {
        'accuracy': accuracy,
        'precision_per_class': precision_pc.tolist(),
        'recall_per_class': recall_pc.tolist(),
        'f1_per_class': f1_pc.tolist(),
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted,
        'f1_weighted': f1_weighted,
        'roc_auc': roc_auc,
        'confusion_matrix': cm.tolist(),
        'confusion_matrix_percent': cm_pct.tolist(),
        'inference_time_ms': float(inf_time),
        'model_size_mb': float(model_size),
        'total_params': int(total_params),
        'y_pred': y_pred.tolist(),
        'y_pred_probs': y_pred_probs.tolist()
    }, history
    
# 11. MAIN
def main():

    print("NS-LSO: Multi-Objective CNN Optimization — RECT Dataset")

    nslso = NSLSO_Optimizer(
        population_size=10,
        max_generations=5,
        hp_range={
            'learning_rate': (1e-4, 5e-3),
            'batch_size':    (16,  128),
            'num_layers':    (2,   4),
            'filters_base':  (8,   64),
            'dropout_rate':  (0.0, 0.5),
            'dense_units':   (16,  256),
            'use_residual':  (0,   1),
            'use_separable': (0,   1)
        }
    )

    start = time.time()
    pareto_front = nslso.optimize(
        x_train_full, y_train_full, x_val_full, y_val_full,
        epochs_per_eval=10, verbose=1
    )
    total_time = time.time() - start

    print("FINAL RESULTS")
    print(f"Total time   : {total_time/3600:.2f} hours")
    print(f"Pareto-front : {len(pareto_front)} solutions")

    pareto_front.sort(key=lambda x: x['accuracy'], reverse=True)
    print("\n--- TOP 5 PARETO SOLUTIONS ---")
    for i, sol in enumerate(pareto_front[:5], 1):
        hp = sol['hyperparams']
        print(f"\n  Solution {i}: acc={sol['accuracy']:.4f} | "
              f"inf={sol['inference_time']:.2f}ms | size={sol['model_size']:.2f}MB")
        print(f"    lr={hp['learning_rate']:.5f}, layers={hp['num_layers']}, "
              f"filters={hp['filters_base']}, batch={hp['batch_size']}, "
              f"dense={hp['dense_units']}, dropout={hp['dropout_rate']:.2f}, "
              f"residual={hp['use_residual']}, separable={hp['use_separable']}")

    print("FINAL EVALUATION ON TEST SET")
    best_hp = pareto_front[0]['hyperparams']
    print(f"\nBest solution val acc : {pareto_front[0]['accuracy']:.4f}")

    results, history = final_test_evaluation(
        best_hp, x_train_full, y_train_full, x_test_full, y_test_full, epochs=30
    )

    print("TEST SET METRICS")
    print(f"\n  Accuracy           : {results['accuracy']:.4f} ({results['accuracy']*100:.2f}%)")
    print(f"  ROC-AUC            : {results['roc_auc']:.4f}")
    print(f"  Precision (macro)  : {results['precision_macro']:.4f}")
    print(f"  Recall (macro)     : {results['recall_macro']:.4f}")
    print(f"  F1-score (macro)   : {results['f1_macro']:.4f}")

    print("\n  --- Per-class metrics ---")
    print(f"  {'Class':<10} {'Precision':>10} {'Recall':>8} {'F1':>8}")
    print("  " + "-"*37)
    for i in range(2):
        print(f"  {CLASS_NAMES[i]:<10}  {results['precision_per_class'][i]:.4f}   "
              f"{results['recall_per_class'][i]:.4f}   {results['f1_per_class'][i]:.4f}")

    print(f"\n  Inference time : {results['inference_time_ms']:.3f} ms/image")
    print(f"  Model size     : {results['model_size_mb']:.2f} MB")
    print(f"  Total params   : {results['total_params']:,}")

if __name__ == "__main__":
    main()

✓ GPUs: 2
GENERATING RECT
Train: (8400, 28, 28, 1) | Val: (1800, 28, 28, 1) | Test: (1800, 28, 28, 1)
Class distribution — Train: [4237 4163] | Val: [903 897] | Test: [872 928]
NS-LSO: Multi-Objective CNN Optimization — RECT Dataset
GENERATION 1/5
  Lion  1/10: acc=1.0000 | inf=2.88ms | size=1.97MB
  Lion  2/10: acc=1.0000 | inf=4.24ms | size=3.15MB
  Lion  3/10: acc=1.0000 | inf=4.29ms | size=3.22MB
  Lion  4/10: acc=0.9833 | inf=5.23ms | size=0.36MB
  Lion  5/10: acc=0.7422 | inf=13.77ms | size=0.37MB
  Lion  6/10: acc=1.0000 | inf=13.51ms | size=0.30MB
  Lion  7/10: acc=1.0000 | inf=3.88ms | size=2.33MB
  Lion  8/10: acc=1.0000 | inf=10.11ms | size=0.74MB
  Lion  9/10: acc=0.9889 | inf=13.22ms | size=0.74MB
  Lion 10/10: acc=1.0000 | inf=4.92ms | size=2.35MB

  >>> FRONT-0: 5 solutions <<<
      acc=1.0000 | inf=2.88ms | size=1.97MB
      acc=0.9833 | inf=5.23ms | size=0.36MB
      acc=1.0000 | inf=13.51ms | size=0.30MB
GENERATION 2/5
  Lion  1/10: acc=0.9511 | inf=13.10ms | size=1.